# Demo del capitolo 3 — audiolab

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 44100
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


## Demo 1 — Calcolare la convoluzione e vedere la coda

In [ ]:
x_breve = np.array([1., 2., 1.])
h_breve = np.array([0.5, 0.5])
y_breve = al.convolve(x_breve, h_breve)
print("Convoluzione completa:", y_breve)
# filter_signal (lfilter) restituisce tanti campioni quanti ne riceve: aggiungiamo la coda.
y_ricorrenza = al.filter_signal(h_breve, [1.], np.pad(x_breve, (0, 1)))
print("Equazione alle differenze:", y_ricorrenza)
fig = make_subplots(rows=1, cols=3, subplot_titles=["Ingresso x", "Risposta h", "Uscita y"])
for col, valori in enumerate((x_breve, h_breve, y_breve), 1):
    fig.add_trace(go.Bar(x=np.arange(len(valori)), y=valori,
                        showlegend=False), row=1, col=col)
fig.update_xaxes(title_text="Campione n", dtick=1)
fig.update_yaxes(title_text="Ampiezza")
fig.update_layout(height=320)
fig.show()


## Demo 2 — Più media significa più attenuazione ovunque?

In [ ]:
fig = go.Figure()
for L in (2, 8, 32):
    b_media = np.ones(L) / L
    f_media, H_media = signal.freqz(b_media, worN=4096, fs=SR)
    fig.add_trace(go.Scatter(x=f_media, y=al.to_db(np.abs(H_media), floor=1e-6),
                            name=f"L = {L}, ordine {L-1}"))
fig.update_xaxes(title_text="Frequenza [Hz]", range=[0, SR/2])
fig.update_yaxes(title_text="Guadagno [dB]", range=[-70, 3])
fig.update_layout(height=380, title="Media mobile: lobo principale e lobi laterali")
fig.show()
chitarra, sr = al.load_audio(AUDIO / "Guitar_C4.wav", sr=SR, seconds=4.0)
versioni_media = [("Originale", np.pad(chitarra, (0, 31)))]
for L in (2, 8, 32):
    filtrata = al.filter_signal(np.ones(L)/L, [1.], np.pad(chitarra, (0, 31)))
    versioni_media.append((f"Media su {L} campioni", filtrata))
al.compare_audio(versioni_media, sr)


## Demo 3 — Una memoria più lunga, una banda più stretta

In [ ]:
impulso = np.zeros(160)
impulso[0] = 1.
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Risposta all'impulso", "Risposta in frequenza"])
for coeff in (0.5, 0.9, 0.98):
    b_polo, a_polo = [1-coeff], [1., -coeff]
    h_polo = al.filter_signal(b_polo, a_polo, impulso)
    f_polo, H_polo = signal.freqz(b_polo, a_polo, worN=4096, fs=SR)
    fig.add_trace(go.Scatter(x=np.arange(len(h_polo)), y=h_polo,
                            name=f"a = {coeff}"), row=1, col=1)
    fig.add_trace(go.Scatter(x=f_polo, y=al.to_db(np.abs(H_polo), floor=1e-6),
                            name=f"a = {coeff}", showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="Campione n", row=1, col=1)
fig.update_yaxes(title_text="Ampiezza", row=1, col=1)
fig.update_xaxes(title_text="Frequenza [Hz]", row=1, col=2)
fig.update_yaxes(title_text="Guadagno [dB]", range=[-50, 3], row=1, col=2)
fig.update_layout(height=380)
fig.show()


## Demo 4 — Collegare il piano z alla risposta in frequenza

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Piano z", "Guadagno"])
angoli = np.linspace(0, 2*np.pi, 400)
fig.add_trace(go.Scatter(x=np.cos(angoli), y=np.sin(angoli),
    mode="lines", line=dict(color="gray", dash="dot"),
    name="Cerchio unitario"), row=1, col=1)
for nome, b_z, a_z in (("Media FIR", [0.5, 0.5], [1., 0.]),
                       ("Media IIR", [0.1, 0.], [1., -0.9])):
    zeri, poli, _ = signal.tf2zpk(b_z, a_z)
    for valori, simbolo, tipo in ((zeri, "circle-open", "zeri"), (poli, "x", "poli")):
        fig.add_trace(go.Scatter(x=valori.real, y=valori.imag, mode="markers",
            marker=dict(symbol=simbolo, size=12), name=f"{nome}: {tipo}"), row=1, col=1)
    f_z, H_z = signal.freqz(b_z, a_z, worN=2048, fs=SR)
    fig.add_trace(go.Scatter(x=f_z, y=al.to_db(np.abs(H_z), floor=1e-6), name=nome), row=1, col=2)
fig.update_xaxes(title_text="Parte reale", range=[-1.2, 1.2], row=1, col=1)
fig.update_yaxes(title_text="Parte immaginaria", range=[-1.2, 1.2],
                 scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_xaxes(title_text="Frequenza [Hz]", row=1, col=2)
fig.update_yaxes(title_text="Guadagno [dB]", range=[-60, 3], row=1, col=2)
fig.update_layout(height=430)
fig.show()


## Demo 5 — Confrontare comb FIR e IIR nel tempo e in frequenza

In [ ]:
D, g = 220, 0.7
b_comb = np.zeros(D+1)
b_comb[0], b_comb[-1] = 1., g
a_comb = np.zeros(D+1)
a_comb[0], a_comb[-1] = 1., -g
imp_comb = np.zeros(10*D+1)
imp_comb[0] = 1.
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Risposte all'impulso", "Risposte in frequenza"])
for nome, b_c, a_c in (("Comb FIR", b_comb, [1.]), ("Comb IIR", [1.], a_comb)):
    h_c = al.filter_signal(b_c, a_c, imp_comb)
    f_c, H_c = signal.freqz(b_c, a_c, worN=16384, fs=SR)
    fig.add_trace(go.Scatter(x=np.arange(len(h_c))/SR*1000, y=h_c,
                            name=nome), row=1, col=1)
    fig.add_trace(go.Scatter(x=f_c, y=al.to_db(np.abs(H_c), floor=1e-6), name=nome,
                            showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="Tempo [ms]", row=1, col=1)
fig.update_yaxes(title_text="Ampiezza", row=1, col=1)
fig.update_xaxes(title_text="Frequenza [Hz]", range=[0, 2000], row=1, col=2)
fig.update_yaxes(title_text="Guadagno [dB]", row=1, col=2)
fig.update_layout(height=380)
fig.show()
print(f"Ritardo: {1000*D/SR:.3f} ms; distanza fra picchi: {SR/D:.2f} Hz")
# Si raccolgono 20 intervalli di coda; l'IIR resta idealmente infinito.
chitarra_c, sr = al.load_audio(AUDIO / "Guitar_C4.wav", sr=SR, seconds=4.0)
x_c = np.pad(chitarra_c, (0, 20*D))
al.compare_audio([("Originale", x_c),
                  ("Comb FIR", al.filter_signal(b_comb, [1.], x_c)),
                  ("Comb IIR", al.filter_signal([1.], a_comb, x_c))], sr)


## Demo 6 — Ascoltare un impulso che diventa un suono risonante

In [ ]:
SR_R, f0 = 10_000, 500
w0 = 2*np.pi*f0/SR_R
imp_r = np.zeros(SR_R)
imp_r[0] = 1.
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Oscillazione smorzata", "Picco di risonanza"])
versioni_r = []
for R in (0.95, 0.99, 0.995):
    G = (1-R)*np.sqrt(1-2*R*np.cos(2*w0)+R**2)
    a_r = [1., -2*R*np.cos(w0), R**2]
    h_r = al.filter_signal([G], a_r, imp_r)
    f_r, H_r = signal.freqz([G], a_r, worN=32768, fs=SR_R)
    fig.add_trace(go.Scatter(x=np.arange(1000)/SR_R, y=h_r[:1000],
                            name=f"R = {R}"), row=1, col=1)
    fig.add_trace(go.Scatter(x=f_r, y=al.to_db(np.abs(H_r), floor=1e-6), name=f"R = {R}",
                            showlegend=False), row=1, col=2)
    versioni_r.append((f"Impulso filtrato, R = {R}", h_r))
    # Misura sulla griglia: approssimazione della larghezza a -3 dB.
    k = np.argmax(np.abs(H_r))
    sopra = np.flatnonzero(np.abs(H_r) >= np.abs(H_r[k])/np.sqrt(2))
    B_mis = f_r[sopra[-1]]-f_r[sopra[0]]
    print(f"R={R}: picco {f_r[k]:.2f} Hz; B misurata {B_mis:.2f} Hz; "
          f"B approssimata {SR_R*(1-R)/np.pi:.2f} Hz")
fig.update_xaxes(title_text="Tempo [s]", row=1, col=1)
fig.update_yaxes(title_text="Ampiezza", row=1, col=1)
fig.update_xaxes(title_text="Frequenza [Hz]", range=[300, 700], row=1, col=2)
fig.update_yaxes(title_text="Guadagno [dB]", range=[-40, 3], row=1, col=2)
fig.update_layout(height=390)
fig.show()
# Guadagno comune esplicito per rendere udibili risposte impulsive deboli.
print("Ascolti: guadagno comune ×8, seguito dall'eventuale attenuazione comune.")
al.compare_audio([(nome, 8*h) for nome, h in versioni_r], SR_R)


## Demo 7 — Dal rumore a un piccolo pop

In [ ]:
SR_R = 10_000
rng = np.random.default_rng(2026)
t_pop = np.arange(SR_R//2)/SR_R
burst = 0.2*rng.uniform(-1, 1, len(t_pop))*np.exp(-t_pop/0.008)
f_pop, B_pop = 400., 80.
R_pop = 1-np.pi*B_pop/SR_R  # progetto approssimato di banda stretta
w_pop = 2*np.pi*f_pop/SR_R
G_pop = (1-R_pop)*np.sqrt(1-2*R_pop*np.cos(2*w_pop)+R_pop**2)
pop = al.filter_signal([G_pop], [1., -2*R_pop*np.cos(w_pop), R_pop**2], burst)
al.compare_audio([("Eccitazione: rumore breve", burst), ("Rumore nel risuonatore", pop)], SR_R)
fig = go.Figure()
for nome, valori in (("Eccitazione", burst), ("Uscita", pop)):
    fig.add_trace(go.Scatter(x=t_pop, y=valori, name=nome))
fig.update_xaxes(title_text="Tempo [s]", range=[0, 0.12])
fig.update_yaxes(title_text="Ampiezza")
fig.update_layout(height=320, title="Un'eccitazione breve mette in moto la risonanza")
fig.show()


## Demo 8 — Un notch e un passa-basso sullo stesso disturbo

In [ ]:
voce, sr = al.load_audio(AUDIO / "speech1.wav", sr=SR, seconds=4.0)
t = np.arange(len(voce))/sr
contaminata = voce + 0.06*np.sin(2*np.pi*2500*t)
sos_notch = al.design_filter("notch", 2500, sr=sr, q=30)
sos_basso = al.design_filter("lowpass", 2000, sr=sr, order=4)
ripulita = al.filter_sos(sos_notch, contaminata)
alternativa = al.filter_sos(sos_basso, contaminata)
versioni = [("Voce di riferimento", voce), ("Con fischio", contaminata),
            ("Notch 2500 Hz", ripulita), ("Passa-basso 2000 Hz", alternativa)]
spettri = [signal.spectrogram(y, fs=sr, window="hann", nperseg=1024,
            noverlap=768, scaling="density") for _, y in versioni]
riferimento = max(float(S.max()) for _, _, S in spettri)
fig = make_subplots(rows=2, cols=2, subplot_titles=[nome for nome, _ in versioni])
for i, (f, tempi, S) in enumerate(spettri):
    fig.add_trace(go.Heatmap(x=tempi, y=f,
        z=10*np.log10(np.maximum(S/riferimento, 1e-8)), coloraxis="coloraxis"),
        row=i//2+1, col=i%2+1)
fig.update_xaxes(title_text="Tempo [s]")
fig.update_yaxes(range=[0, 6000], title_text="Frequenza [Hz]")
fig.update_layout(height=600, coloraxis=dict(colorscale="Magma", cmin=-70, cmax=0,
    colorbar=dict(title="PSD relativa [dB]")), title="Stessa analisi e stessa scala per tutte le versioni")
fig.show()
al.compare_audio(versioni, sr)


## Demo 9 — Progettare e verificare le due campane EQ (Filter A e Filter B)

In [ ]:
fig = go.Figure()
for f0_eq, B_eq, gain_eq, GB_eq in ((1700, 500, 9, 6), (2600, 1000, -9, -6)):
    b_eq, a_eq = al.eq_bell(f0_eq, B_eq, gain_eq, sr=10000, band_level_db=GB_eq)
    f_eq, H_eq = signal.freqz(b_eq, a_eq, worN=65536, fs=10000)
    db_eq = al.to_db(np.abs(H_eq), floor=1e-6)
    selezione = db_eq >= GB_eq if gain_eq > 0 else db_eq <= GB_eq
    indici = np.flatnonzero(selezione)
    B_mis = f_eq[indici[-1]]-f_eq[indici[0]]
    _, al_centro = signal.freqz(b_eq, a_eq, worN=[f0_eq], fs=10000)
    print(f"Centro {f0_eq} Hz: {al.to_db(np.abs(al_centro), floor=1e-6)[0]:.3f} dB; "
          f"banda misurata a {GB_eq:+} dB: {B_mis:.2f} Hz")
    fig.add_trace(go.Scatter(x=f_eq, y=db_eq, name=f"{gain_eq:+} dB a {f0_eq} Hz"))
    fig.add_trace(go.Scatter(x=f_eq[indici[[0, -1]]], y=db_eq[indici[[0, -1]]],
        mode="markers", marker=dict(size=9), name=f"Estremi a {GB_eq:+} dB"))
fig.update_xaxes(title_text="Frequenza [Hz]")
fig.update_yaxes(title_text="Guadagno [dB]", range=[-12, 12])
fig.update_layout(height=380, title="Specifiche e risposta della cella parametrica")
fig.show()


## Demo 10 — Ascoltare campana, mensola e passa-basso

In [ ]:
b_sh, a_sh = al.eq_shelf(2500, -9, "high")
b_bell, a_bell = al.eq_bell(2500, 1000, -9)
sos_lp = al.design_filter("lowpass", 2500, sr=SR, order=4)
progetti = {
    "Campana −9 dB": signal.tf2sos(b_bell, a_bell),
    "Mensola alte −9 dB": signal.tf2sos(b_sh, a_sh),
    "Passa-basso 2500 Hz": sos_lp,
}
fig = go.Figure()
voce_eq, sr = al.load_audio(AUDIO / "speech1.wav", sr=SR, seconds=4.0)
versioni_eq = [("Originale", voce_eq)]
f_griglia = np.geomspace(30, 20000, 2000)
for nome, sos_eq in progetti.items():
    _, H_eq = signal.sosfreqz(sos_eq, worN=f_griglia, fs=sr)
    fig.add_trace(go.Scatter(x=f_griglia, y=al.to_db(np.abs(H_eq), floor=1e-6), name=nome))
    versioni_eq.append((nome, al.filter_sos(sos_eq, voce_eq)))
fig.update_xaxes(type="log", title_text="Frequenza [Hz]")
fig.update_yaxes(title_text="Guadagno [dB]", range=[-45, 3])
fig.update_layout(height=380, title="Stessa frequenza indicata, tre interventi diversi")
fig.show()
al.compare_audio(versioni_eq, sr)


## Demo 11 — Equalizzazione in cascata a 4 bande su parlato (speech.wav) e musica (acoustic.wav)

In [ ]:
# 1. Catena a 4 bande per il parlato (speech.wav)
voce_sp, sr_sp = al.load_audio(AUDIO / "speech.wav", sr=SR, seconds=5.0)
stadi_speech = [
    ("Banda 1: LP Shelf -8 dB (200 Hz)", signal.tf2sos(*al.eq_shelf(200, -8.0, "low", sr=sr_sp))),
    ("Banda 2: Cut -6 dB (420 Hz)", signal.tf2sos(*al.eq_bell(420, 180, -6.0, sr=sr_sp, band_level_db=-3.0))),
    ("Banda 3: Boost +7 dB (2.8 kHz)", signal.tf2sos(*al.eq_bell(2800, 1200, 7.0, sr=sr_sp, band_level_db=3.5))),
    ("Banda 4: HP Shelf +4 dB (6 kHz)", signal.tf2sos(*al.eq_shelf(6000, 4.0, "high", sr=sr_sp))),
]
sos_speech_tot = np.vstack([sos for _, sos in stadi_speech])
voce_sp_eq = al.filter_sos(sos_speech_tot, voce_sp)
voce_sp_eq_rms = voce_sp_eq * (al.rms(voce_sp) / max(al.rms(voce_sp_eq), 1e-12))

# 2. Catena a 4 bande per la musica (acoustic.wav)
musica_ac, sr_ac = al.load_audio(AUDIO / "acoustic.wav", sr=SR, seconds=6.0)
stadi_musica = [
    ("Banda 1: LP Shelf +6.5 dB (140 Hz)", signal.tf2sos(*al.eq_shelf(140, 6.5, "low", sr=sr_ac))),
    ("Banda 2: Cut -7.5 dB (850 Hz)", signal.tf2sos(*al.eq_bell(850, 450, -7.5, sr=sr_ac, band_level_db=-3.8))),
    ("Banda 3: Boost +5.5 dB (3.5 kHz)", signal.tf2sos(*al.eq_bell(3500, 1500, 5.5, sr=sr_ac, band_level_db=2.8))),
    ("Banda 4: HP Shelf +6 dB (8 kHz)", signal.tf2sos(*al.eq_shelf(8000, 6.0, "high", sr=sr_ac))),
]
sos_musica_tot = np.vstack([sos for _, sos in stadi_musica])
musica_ac_eq = al.filter_sos(sos_musica_tot, musica_ac)
musica_ac_eq_rms = musica_ac_eq * (al.rms(musica_ac) / max(al.rms(musica_ac_eq), 1e-12))

f_casc = np.geomspace(30, 18000, 1500)
fig = make_subplots(rows=1, cols=2, subplot_titles=["Curva EQ Speech (speech.wav)", "Curva EQ Musica (acoustic.wav)"])
for col, (stadi, sos_tot, sr_cur) in enumerate(((stadi_speech, sos_speech_tot, sr_sp),
                                                (stadi_musica, sos_musica_tot, sr_ac)), 1):
    f_lim = f_casc[f_casc < 0.49 * sr_cur]
    for nome, sos in stadi:
        _, H = signal.sosfreqz(sos, worN=f_lim, fs=sr_cur)
        fig.add_trace(go.Scatter(x=f_lim, y=al.to_db(np.abs(H), floor=1e-6), name=nome,
                                 line=dict(dash="dot"), showlegend=(col == 1)), row=1, col=col)
    _, H_tot = signal.sosfreqz(sos_tot, worN=f_lim, fs=sr_cur)
    fig.add_trace(go.Scatter(x=f_lim, y=al.to_db(np.abs(H_tot), floor=1e-6), name=f"Totale (col {col})",
                             line=dict(width=3)), row=1, col=col)
fig.update_xaxes(type="log", title_text="Frequenza [Hz]")
fig.update_yaxes(title_text="Guadagno [dB]", range=[-12, 10])
fig.update_layout(height=390)
fig.show()

print("Confronto 1 — Parlato (speech.wav):")
al.compare_audio([("Speech originale", voce_sp),
                  ("Speech equalizzato (4 bande)", voce_sp_eq_rms)], sr_sp)
print("Confronto 2 — Musica (acoustic.wav):")
al.compare_audio([("Musica originale", musica_ac),
                  ("Musica equalizzata (Warmth & Air)", musica_ac_eq_rms)], sr_ac)


## Demo 12 — Una sola copia e un eco con feedback

In [ ]:
nota, sr = al.load_audio(AUDIO / "Guitar_C4.wav", sr=SR, seconds=1.2)
x_echo = np.pad(nota, (0, round(3*sr)))
una_copia = al.delay_signal(0.7 * x_echo, 0.375, gain=0.3, sr=sr)[:len(x_echo)]
ripetizioni = al.delay_by(al.feedback_delay(x_echo, 0.375, feedback=0.55, sr=sr), 0.375, sr=sr)[:len(x_echo)]
eco = 0.7 * x_echo + 0.3 * ripetizioni
al.compare_audio([("Diretto", x_echo), ("Una copia", una_copia),
                  ("Ripetizioni con feedback", eco)], sr)
fig = go.Figure()
passo = 40
fig.add_trace(go.Scatter(x=np.arange(0, len(eco), passo)/sr, y=eco[::passo],
                        name="Diretto + ripetizioni"))
fig.add_vline(x=len(nota)/sr, line_dash="dot", annotation_text="Fine ingresso")
fig.update_xaxes(title_text="Tempo [s]")
fig.update_yaxes(title_text="Ampiezza")
fig.update_layout(height=320, title="Delay di una croma puntata a 120 BPM")
fig.show()
print(f"Picco negli ultimi 200 ms: {al.to_db(al.peak_rms(eco[-sr//5:])[0]):.1f} dBFS")


## Demo 13 — Implementare $d(n)$ per Flanger e Chorus su chitarra acustica

In [ ]:
chitarra_ac, sr = al.load_audio(AUDIO / "acoustic.wav", sr=SR, seconds=5.0)
n_idx = np.arange(len(chitarra_ac))
t_ac = n_idx / sr

# 1. Flanger: d(n) = (D/2) * (1 - cos(2*pi*fd*n/Fs)), con D = 5 ms, fd = 1 Hz, a = 0.8
D_fl_max = 0.005 * sr
fd_fl = 1.0
wet_flanger, d_flanger = al.modulated_delay(chitarra_ac, sr, delay_ms=2.5, depth_ms=2.5, rate_hz=fd_fl, phase=-np.pi/2)
y_flanger = (chitarra_ac + 0.8 * wet_flanger) / 1.8

# 2. Chorus: d_k(n) = D1 + (D2 - D1) * (0.5 + r_k(n)), con r_k(n) casuale lento in [-0.5, 0.5)
rng_ch = np.random.default_rng(42)
b_lp_lfo, a_lp_lfo = signal.butter(2, 2.5 / (sr / 2.0), btype="low")
D1_ch, D2_ch = 0.010 * sr, 0.025 * sr
d_chorus_voci = []
y_chorus = chitarra_ac.copy()
for voce_k in range(2):
    rumore_lento = signal.filtfilt(b_lp_lfo, a_lp_lfo, rng_ch.uniform(-1.0, 1.0, len(chitarra_ac)))
    r_k = 0.48 * rumore_lento / max(np.max(np.abs(rumore_lento)), 1e-12)
    d_k = D1_ch + (D2_ch - D1_ch) * (0.5 + r_k)
    d_chorus_voci.append(d_k)
    wet_voce, _ = al.modulated_delay(chitarra_ac, sr, delay_curve=d_k)
    y_chorus = y_chorus + 0.6 * wet_voce
y_chorus = y_chorus / 2.2

al.compare_audio([("Originale (acoustic.wav)", chitarra_ac),
                  ("Flanger (sinusoidale 0–5 ms, 1 Hz)", y_flanger),
                  ("Chorus (2 voci casuali 10–25 ms)", y_chorus)], sr)

fig = go.Figure()
passo_plot = 100
fig.add_trace(go.Scatter(x=t_ac[::passo_plot], y=1000 * d_flanger[::passo_plot] / sr,
                         name="Flanger: d(n) sinusoidale"))
for idx_v, d_k in enumerate(d_chorus_voci, 1):
    fig.add_trace(go.Scatter(x=t_ac[::passo_plot], y=1000 * d_k[::passo_plot] / sr,
                             name=f"Chorus: voce {idx_v} casuale lenta"))
fig.update_xaxes(title_text="Tempo [s]")
fig.update_yaxes(title_text="Ritardo d(n) [ms]")
fig.update_layout(height=360, title="Confronto tra le leggi di ritardo d(n) di Flanger e Chorus")
fig.show()
